In [5]:
import math
import unicodedata

# ---------------------------------------------------------------------
# dados
# ---------------------------------------------------------------------

TARIFA = 1.15           # R$/kWh com tributos - ESTIMADO (ANEEL: R$ 0,851 sem tributos / (1 - 0,26))
REAJUSTE = 0.04         # reajuste anual da tarifa - ESTIMADO
FATOR_CO2 = 0.0385      # kg CO2 por kWh - oficial (MCTI, média do SIN em 2023: 38,5 kg/MWh)
DISPONIBILIDADE = 30    # kWh/mês pagos mesmo com solar (monofásico) - conferir na ANEEL
PERIODO = 10            # período de comparação (anos), igual para todas as alternativas

BANDEIRA_ATUAL = "verde"  # oficial: verde em novembro/2026
BANDEIRAS = {"verde": 0.0, "amarela": 1.885, "vermelha 1": 4.46, "vermelha 2": 7.87}  # R$ por 100 kWh

# energia solar (tudo ESTIMADO)
HSP = 4.5               # horas de sol pleno por dia (kwh/m²/dia)
DESEMPENHO = 0.80       # perdas do sistema
MODULO_KWP = 0.55       # cada módulo = 550 Wp
DEGRADACAO = 0.005      # perda de geração por ano
FATOR_COMP = 0.90       # valor do kWh gerado frente ao comprado (Fio B)
MANUTENCAO = 0.005      # manutenção por ano (% do investimento)
CUSTO_WP = {"otimista": 3.5, "base": 4.5, "pessimista": 6.0}  # R$ por Wp instalado

# casa de referencia: família de 4 pessoas
# (nome, quantidade, potência W, horas/dia, dias/mês, kwh/mês fixo, tipo do dado)
EQUIPAMENTOS = [
    ("Geladeira 2 portas frost free", 1, None, None, None, 56.88, "oficial (Procel)"),
    ("Chuveiro elétrico 5500 W", 1, 5500, 0.67, 30, None, "estimado"),
    ("Lâmpadas fluorescentes 20 W", 8, 20, 5, 30, None, "estimado"),
    ("TV 40 polegadas", 1, 100, 5, 30, None, "estimado"),
    ("Ventilador 80 W", 2, 80, 6, 30, None, "estimado"),
    ("Máquina de lavar roupa", 1, 500, 1, 12, None, "estimado"),
    ("Ferro de passar", 1, 1000, 1, 8, None, "estimado"),
    ("Roteador wi-fi", 1, 10, 24, 30, None, "estimado"),
    ("Standby e carregadores", 1, 15, 24, 30, None, "estimado"),
    ("Micro-ondas", 1, 1200, 0.25, 30, None, "estimado"),
]


# ---------------------------------------------------------------------
# FUNÇÕES AUXILIARES
# ---------------------------------------------------------------------

def rs(v):
    s = f"{v:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return "R$ " + s


def num(v, casas=1):
    return f"{v:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def tempo(anos):
    if anos is None:
        return "não se paga em 10 anos"
    if anos < 1:
        m = max(1, round(anos * 12))
        return f"cerca de {m} {'mês' if m == 1 else 'meses'}"
    return f"cerca de {num(anos)} anos"


def sem_acento(texto):
    base = unicodedata.normalize("NFKD", texto.lower())
    return "".join(c for c in base if not unicodedata.combining(c))


# ---------------------------------------------------------------------
# consumo: energia (kWh) = potência (kW) x tempo (h)
# ---------------------------------------------------------------------

def consumo_item(item):
    nome, qtd, watts, horas, dias, fixo, tipo = item
    if fixo is not None:
        return fixo * qtd
    return (watts / 1000) * horas * dias * qtd


def consumo_total(equipamentos):
    return sum(consumo_item(i) for i in equipamentos)


def ranking(equipamentos):
    total = consumo_total(equipamentos)
    linhas = [(i[0], consumo_item(i), consumo_item(i) / total * 100) for i in equipamentos]
    return sorted(linhas, key=lambda x: x[1], reverse=True)


def custo_mensal(kwh, bandeira=BANDEIRA_ATUAL):
    return kwh * TARIFA + (kwh / 100) * BANDEIRAS[bandeira]


def co2(kwh):
    return kwh * FATOR_CO2


def payback(serie):
    """Anos até o saldo acumulado ficar positivo (interpolação linear)."""
    for ano in range(1, len(serie)):
        if serie[ano] >= 0 > serie[ano - 1]:
            return ano - 1 + (-serie[ano - 1]) / (serie[ano] - serie[ano - 1])
    return None


# ---------------------------------------------------------------------
# opção a: eficiência energética (LED + banho menor + standby)
# ---------------------------------------------------------------------
def aplicar_eficiencia(equipamentos, led=True, banho=True, standby=True):
    novos, custo = [], 0.0
    for nome, qtd, watts, horas, dias, fixo, tipo in equipamentos:
        if led and nome.startswith("Lâmpadas"):
            watts = 9                    # LED de 9 W no lugar de 20 W
            custo += qtd * 8.0           # R$ 8 por lâmpada (ESTIMADO)
        if banho and nome.startswith("Chuveiro"):
            horas = horas * 0.75         # banho 25% mais curto (10 -> 7,5 min)
        if standby and nome.startswith("Standby"):
            watts = watts * 0.5          # régua de tomadas desliga metade
            custo += 25.0                # R$ 25 (ESTIMADO)
        novos.append((nome, qtd, watts, horas, dias, fixo, tipo))
    return novos, custo


def simular_eficiencia(equipamentos, led=True, banho=True, standby=True):
    novos, capex = aplicar_eficiencia(equipamentos, led, banho, standby)
    antes, depois = consumo_total(equipamentos), consumo_total(novos)
    economia = antes - depois
    acum, serie = -capex, [-capex]
    for ano in range(PERIODO):
        acum += economia * 12 * TARIFA * (1 + REAJUSTE) ** ano
        serie.append(acum)
    kwh_periodo = economia * 12 * PERIODO
    return {
        "capex": capex, "kwh_antes": antes, "kwh_depois": depois,
        "economia_kwh": economia, "economia_rs_mes": economia * TARIFA,
        "serie": serie, "saldo": serie[-1], "payback": payback(serie),
        "kwh_periodo": kwh_periodo, "co2": co2(kwh_periodo),
        "custo_kwh": capex / kwh_periodo if kwh_periodo else 0.0,
    }


# ---------------------------------------------------------------------
# opção b: energia solar fotovoltaica (renovável)
# geração mensal = kwp x hsp x desempenho x 30 dias
# ---------------------------------------------------------------------
def simular_solar(consumo, cenario="base", reajuste=REAJUSTE, tarifa=TARIFA):
    ger_por_kwp = HSP * 30 * DESEMPENHO
    alvo = max(consumo - DISPONIBILIDADE, 0)
    modulos = math.ceil(alvo / ger_por_kwp / MODULO_KWP)
    kwp = modulos * MODULO_KWP
    capex = kwp * 1000 * CUSTO_WP[cenario]
    geracao = kwp * ger_por_kwp
    acum, serie, kwh_periodo, gasto = -capex, [-capex], 0.0, capex
    for ano in range(PERIODO):
        comp = min(geracao * (1 - DEGRADACAO) ** ano, alvo)
        acum += comp * 12 * tarifa * (1 + reajuste) ** ano * FATOR_COMP - capex * MANUTENCAO
        gasto += capex * MANUTENCAO
        kwh_periodo += comp * 12
        serie.append(acum)
    return {
        "kwp": kwp, "modulos": modulos, "capex": capex, "geracao": geracao,
        "economia_rs_mes": min(geracao, alvo) * tarifa * FATOR_COMP,
        "serie": serie, "saldo": serie[-1], "payback": payback(serie),
        "kwh_periodo": kwh_periodo, "co2": co2(kwh_periodo),
        "custo_kwh": gasto / kwh_periodo if kwh_periodo else 0.0,
    }


def simular_combinado(equipamentos):
    """Primeiro reduz o consumo, depois dimensiona um solar menor."""
    ef = simular_eficiencia(equipamentos)
    so = simular_solar(ef["kwh_depois"])
    serie = [a + b for a, b in zip(ef["serie"], so["serie"])]
    return {
        "kwp": so["kwp"], "capex": ef["capex"] + so["capex"],
        "economia_rs_mes": ef["economia_rs_mes"] + so["economia_rs_mes"],
        "serie": serie, "saldo": serie[-1], "payback": payback(serie),
        "co2": ef["co2"] + so["co2"],
    }


# ---------------------------------------------------------------------
# 6. assistente (responde usando os calculos na parte de cima)
# ---------------------------------------------------------------------

AVISO = ("Estimativas, não garantias: dependem da tarifa, do uso dos aparelhos, do sol da sua "
         "região e do preço do sistema. Confira sua conta de luz e peça orçamentos antes de decidir.")


def com_rodape(texto, base):
    return f"{texto}\n\nBase do cálculo: {base}\n{AVISO}"


def extrair_consumo(pergunta):
    """Procura um número seguido de kWh (ex.: 'minha conta é 300 kWh')."""
    import re
    m = re.search(r"(\d+(?:[.,]\d+)?)\s*kwh", sem_acento(pergunta))
    return float(m.group(1).replace(",", ".")) if m else None


def responder(pergunta):
    t = sem_acento(pergunta)
    total = consumo_total(EQUIPAMENTOS)

    if "tarifa social" in t:
        return com_rodape(
            "A Tarifa Social dá desconto na conta de luz de famílias de baixa renda. Pela regra em vigor "
            "para faturas a partir de 5/7/2025, o desconto é de 100% até 80 kWh por mês e não há desconto "
            "acima disso. Em geral, é preciso estar no CadÚnico (atualizado nos últimos 2 anos) com renda "
            "de até meio salário mínimo por pessoa, ou receber o BPC. Peça o benefício na sua distribuidora "
            "ou procure o CRAS.",
            "página da ANEEL sobre Tarifa Social (gov.br/aneel). As regras podem mudar; confirme na distribuidora.")

    if "bandeira" in t:
        linhas = [f"Bandeira tarifária é um valor extra na conta quando a energia fica mais cara. "
                  f"Para {num(total, 0)} kWh/mês, o extra seria:"]
        for nome, add in BANDEIRAS.items():
            linhas.append(f"- {nome}: {rs(total / 100 * add)}")
        linhas.append(f"Hoje a bandeira é {BANDEIRA_ATUAL}.")
        return com_rodape("\n".join(linhas), "adicionais da ANEEL por 100 kWh.")

    if "fonte" in t or "referencia" in t:
        return ("Fontes: ANEEL (tarifa, bandeiras), Procel (consumo da geladeira), MCTI (emissão de CO2), "
                "Greener e Monitor do Mercado (preço do solar). Dados 'estimados' estão marcados no código.")

    if "limita" in t or "garant" in t:
        return ("Limitações: a casa de referência é hipotética (4 pessoas); a tarifa de R$ 1,15/kWh, o preço "
                "do solar e o sol local são estimativas; não incluímos financiamento nem sombreamento; "
                "leis (Fio B, tarifa social) podem mudar os resultados.\n" + AVISO)

    if "compar" in t or "recomend" in t or "qual vale" in t:
        ef, so, cb = simular_eficiencia(EQUIPAMENTOS), simular_solar(total), simular_combinado(EQUIPAMENTOS)
        txt = (f"Comparação para a casa de referência ({num(total, 0)} kWh/mês), em {PERIODO} anos:\n"
               f"- A) Eficiência: investe {rs(ef['capex'])}, economiza {rs(ef['economia_rs_mes'])}/mês, "
               f"retorno em {tempo(ef['payback'])}, saldo {rs(ef['saldo'])}.\n"
               f"- B) Solar ({num(so['kwp'], 2)} kWp): investe {rs(so['capex'])}, economiza "
               f"{rs(so['economia_rs_mes'])}/mês, retorno em {tempo(so['payback'])}, saldo {rs(so['saldo'])}.\n"
               f"- A+B) Combinado ({num(cb['kwp'], 2)} kWp): investe {rs(cb['capex'])}, economiza "
               f"{rs(cb['economia_rs_mes'])}/mês, retorno em {tempo(cb['payback'])}.\n"
               "Recomendação: comece pela eficiência, que quase não custa. Se houver telhado e "
               "financiamento, avalie o solar depois de reduzir o consumo.")
        return com_rodape(txt, "mesma necessidade (reduzir o custo de energia) e mesmo período de 10 anos.")

    if "solar" in t or "painel" in t or "placa" in t or "fotovolt" in t:
        consumo = extrair_consumo(pergunta) or total
        if consumo <= DISPONIBILIDADE + 20:
            return (f"Com {num(consumo, 0)} kWh/mês quase tudo já é a taxa mínima de {DISPONIBILIDADE} kWh, "
                    "paga mesmo com painéis. O solar tende a não compensar.")
        r = simular_solar(consumo)
        pior = simular_solar(consumo, "pessimista", reajuste=0.0)
        txt = (f"Para {num(consumo, 0)} kWh/mês, o sistema estimado é de {num(r['kwp'], 2)} kWp "
               f"({r['modulos']} módulos), gerando ~{num(r['geracao'], 0)} kWh/mês.\n"
               f"- Investimento (cenário base): {rs(r['capex'])}\n"
               f"- Economia: ~{rs(r['economia_rs_mes'])} por mês\n"
               f"- Retorno: {tempo(r['payback'])} (no pior cenário testado: {tempo(pior['payback'])})\n"
               f"- Saldo em 10 anos: {rs(r['saldo'])}\n"
               f"- CO2 evitado em 10 anos: ~{num(r['co2'], 0)} kg (pouco, pois a rede brasileira já é limpa)\n"
               "O investimento inicial é alto para famílias de baixa renda; reduza o consumo antes.")
        return com_rodape(txt, "HSP 4,5; desempenho 80%; R$ 4,50/Wp (estimado); reajuste de 4%/ano.")

    if "led" in t or "lampada" in t:
        r = simular_eficiencia(EQUIPAMENTOS, True, False, False)
        txt = (f"Trocar as 8 lâmpadas de 20 W por LED de 9 W economiza {num(r['economia_kwh'])} kWh/mês "
               f"({rs(r['economia_rs_mes'])}/mês). Custa cerca de {rs(r['capex'])} e se paga em "
               f"{tempo(r['payback'])}.")
        return com_rodape(txt, "8 lâmpadas, 5 h/dia, R$ 8 cada (estimado).")

    if "banho" in t or "chuveiro" in t:
        r = simular_eficiencia(EQUIPAMENTOS, False, True, False)
        txt = (f"Reduzir o banho de 10 para 7,5 minutos por pessoa economiza {num(r['economia_kwh'])} kWh/mês "
               f"({rs(r['economia_rs_mes'])}/mês), sem gastar nada.")
        return com_rodape(txt, "chuveiro de 5500 W, 4 pessoas, 10 min cada (estimado).")

    if "co2" in t or "ambient" in t or "carbono" in t or "sustent" in t:
        ef, so = simular_eficiencia(EQUIPAMENTOS), simular_solar(total)
        txt = (f"A rede elétrica brasileira emite pouco CO2 (~{num(FATOR_CO2 * 1000)} kg por MWh), então o "
               f"ganho ambiental é modesto: eficiência evita ~{num(ef['co2'], 0)} kg e solar ~{num(so['co2'], 0)} kg "
               f"em 10 anos. O ganho principal para a família é financeiro.")
        return com_rodape(txt, "fator de emissão do MCTI (2023); sem contar a fabricação dos equipamentos.")

    if "gasta" in t or "consome" in t or "aparelho" in t or "consumo" in t or "economizar" in t:
        linhas = [f"Na casa de referência o consumo é de {num(total, 0)} kWh/mês "
                  f"(cerca de {rs(custo_mensal(total))}). Os que mais gastam:"]
        for nome, kwh, pct in ranking(EQUIPAMENTOS)[:4]:
            linhas.append(f"- {nome}: {num(kwh, 0)} kWh/mês ({num(pct, 0)}%, {rs(kwh * TARIFA)})")
        return com_rodape("\n".join(linhas), "potência x horas de uso (kWh) x tarifa de R$ 1,15/kWh (estimada).")

    if "ajuda" in t or t in ("oi", "ola", "menu"):
        return ("Posso responder: quais aparelhos gastam mais; LED; banho; energia solar (ex.: 'solar, "
                "minha conta é 300 kWh'); compare as opções; bandeira tarifária; tarifa social; "
                "impacto ambiental; fontes; limitações.")

    return "Não entendi. Digite 'ajuda' para ver o que eu sei responder."


# ---------------------------------------------------------------------
# execução: tabela da análise + conversa com o assistente
# ---------------------------------------------------------------------
def mostrar_analise():
    total = consumo_total(EQUIPAMENTOS)
    print(f"CASA DE REFERÊNCIA: {num(total)} kWh/mês = {rs(custo_mensal(total))}/mês\n")
    print(f"{'Equipamento':34}{'kWh/mês':>9}{'%':>7}  Dado")
    tipos = {i[0]: i[6] for i in EQUIPAMENTOS}
    for nome, kwh, pct in ranking(EQUIPAMENTOS):
        print(f"{nome:34}{num(kwh):>9}{num(pct, 0):>6}%  {tipos[nome]}")

    ef, so, cb = simular_eficiencia(EQUIPAMENTOS), simular_solar(total), simular_combinado(EQUIPAMENTOS)
    print(f"\nCOMPARAÇÃO EM {PERIODO} ANOS")
    print(f"{'':26}{'A) Eficiência':>16}{'B) Solar':>16}{'A+B':>16}")
    print(f"{'Investimento':26}{rs(ef['capex']):>16}{rs(so['capex']):>16}{rs(cb['capex']):>16}")
    print(f"{'Economia por mês':26}{rs(ef['economia_rs_mes']):>16}{rs(so['economia_rs_mes']):>16}{rs(cb['economia_rs_mes']):>16}")
    print(f"{'Retorno (anos)':26}{num(ef['payback'], 1):>16}{num(so['payback'], 1):>16}{num(cb['payback'], 1):>16}")
    print(f"{'Saldo em 10 anos':26}{rs(ef['saldo']):>16}{rs(so['saldo']):>16}{rs(cb['saldo']):>16}")
    print(f"{'CO2 evitado (kg)':26}{num(ef['co2'], 0):>16}{num(so['co2'], 0):>16}{num(cb['co2'], 0):>16}")

    print("\nSENSIBILIDADE DO SOLAR (preço do sistema x reajuste da tarifa)")
    for cenario in CUSTO_WP:
        for reaj in (0.0, REAJUSTE):
            r = simular_solar(total, cenario, reaj)
            print(f"  {cenario:11} R$ {CUSTO_WP[cenario]:.2f}/Wp, reajuste {reaj * 100:.0f}%: "
                  f"investimento {rs(r['capex'])}, retorno {num(r['payback'], 1)} anos")


def conversar():
    print("\nAssistente de energia. Digite 'ajuda' ou 'sair'.")
    while True:
        try:
            pergunta = input("\nVocê: ").strip()
        except EOFError:
            break
        if not pergunta:
            continue
        if sem_acento(pergunta) in ("sair", "exit"):
            break
        print("\nAssistente:", responder(pergunta))


if __name__ == "__main__":
    mostrar_analise()
    conversar()

CASA DE REFERÊNCIA: 276,2 kWh/mês = R$ 317,66/mês

Equipamento                         kWh/mês      %  Dado
Chuveiro elétrico 5500 W              110,5    40%  estimado
Geladeira 2 portas frost free          56,9    21%  oficial (Procel)
Ventilador 80 W                        28,8    10%  estimado
Lâmpadas fluorescentes 20 W            24,0     9%  estimado
TV 40 polegadas                        15,0     5%  estimado
Standby e carregadores                 10,8     4%  estimado
Micro-ondas                             9,0     3%  estimado
Ferro de passar                         8,0     3%  estimado
Roteador wi-fi                          7,2     3%  estimado
Máquina de lavar roupa                  6,0     2%  estimado

COMPARAÇÃO EM 10 ANOS
                             A) Eficiência        B) Solar             A+B
Investimento                      R$ 89,00    R$ 12.375,00     R$ 9.989,00
Economia por mês                  R$ 53,17       R$ 254,85       R$ 260,17
Retorno (anos)            